In [1]:
import torch
import json
from pathlib import Path
from pydantic import BaseModel
from torch.utils.data import Dataset
from utils import json_load, json_dump
from data.data import img_read

In [2]:
json_load("/hpc/jtia475/datasets/mytest/data_set.json")

{'sub-strokecase0140_ses-0001': {'img_path': {'adc': '/hpc/jtia475/datasets/ISLES-2022/ADC/sub-strokecase0140_ses-0001_adc.nii.gz',
   'flair': '/hpc/jtia475/datasets/ISLES-2022/FLAIR/sub-strokecase0140_ses-0001_FLAIR.nii.gz',
   'dwi': '/hpc/jtia475/datasets/ISLES-2022/DWI/sub-strokecase0140_ses-0001_dwi.nii.gz',
   'msk': '/hpc/jtia475/datasets/ISLES-2022/MASK/sub-strokecase0140_ses-0001_msk.nii.gz'},
  'text_label': {'global_finding': 'Scattered patchy abnormal signal shadows are seen in the brainstem, bilateral cerebellum, bilateral occipital lobes, left thalamus, and parietal lobes, showing low signal on ADC sequence, high signal on DWI sequence, with no midline shift.',
   'impression': {'disease': ['infarct foci (subacute stage)'],
    'impression': 'Scattered infarct foci in the brainstem, bilateral cerebellum, bilateral occipital lobes, left thalamus, and parietal lobes (subacute stage), please correlate clinically.'},
   'modal_wise_finding': {'adc': 'Lesions appear as low si

In [16]:
class MyDataset(Dataset):
    def __init__(self, data_path):
        self.data = list(json_load(data_path).values())

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        img_path = item["img_path"]["adc"]
        img = img_read(img_path,(3,1,1))

        text = item["text_label"]["modal_wise_finding"]["adc"]
        return img, text


dataset = MyDataset("/hpc/jtia475/datasets/mytest/data_set.json")
img, prompt, target = dataset[0]  # Access the first item in the dataset
img.shape

(48, 224, 224)

In [17]:
from models.vlm import VLM, VlmConfig

cfg = json_load("/hpc/jtia475/vlm_train/configs/test.json")

cfg = VlmConfig(**cfg["vlm"])



In [18]:
model = VLM(cfg)

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

In [ ]:
x = torch.tensor(img).unsqueeze(0).unsqueeze(0)
x_prompt = [prompt]
x_target = [target]
y = model(x, prompt=x_prompt, target=x_target)

In [27]:
y[0].shape, y[1].shape

(torch.Size([1, 144, 151669]), torch.Size([]))

In [28]:
model.generate(x, prompt=prompt)

["</think>\n\n</think>\n\nOkay, let's see. The user provided the query. The user is asking for the analysis. The user wants the analysis. The user wants the analysis. The user wants the analysis. The user wants the analysis. The user wants the analysis. The user wants the analysis. The user wants the analysis. The user wants the analysis. The user wants the analysis. The user wants the analysis. The user wants the analysis. The user wants the analysis. The user wants the analysis. The user wants the analysis. The user wants the analysis. The user wants the analysis. The user wants the analysis. The user"]